In [1]:
import os
import copy
import numpy as np

import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR

from train_utils import (
    sample_mask_uniform_K_per_sample,
    run_feature_acquisition
)

from 쪼인트 import (
    AcquisitionModel,
    PredictionModel,
    ModelLoss,
)

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # blackwell

# 데이터 

X_train = torch.load(f"/home/sulee/Gamma-CMI/data/metabric/X_train_cdf.pt").float()
y_train = torch.load(f"/home/sulee/Gamma-CMI/data/metabric/y_train.pt").long()

X_val   = torch.load(f"/home/sulee/Gamma-CMI/data/metabric/X_val_cdf.pt").float()
y_val   = torch.load(f"/home/sulee/Gamma-CMI/data/metabric/y_val.pt").long()

X_test  = torch.load(f"/home/sulee/Gamma-CMI/data/metabric/X_test_cdf.pt").float()
y_test  = torch.load(f"/home/sulee/Gamma-CMI/data/metabric/y_test.pt").long()

num_features = 12
num_classes  = 6

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 256

train_ds = TensorDataset(X_train, y_train)
val_ds   = TensorDataset(X_val,   y_val)
test_ds  = TensorDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

In [3]:
# 모델 정의

z_dim          = 64    # per-feature latent dim
enc_hidden_dim = 8     # JointEncoder 내부 attention token dim
dec_hidden_dim = 256   # decoder MLP hidden dim
dec_num_hidden = 2     # decoder MLP hidden layer 개수
num_heads      = 8     # attention head의 개수

acquisition_model = AcquisitionModel(
    num_features=num_features,
    z_dim=z_dim,
    enc_hidden_dim=enc_hidden_dim,
    num_heads=num_heads,
    dec_hidden_dim=dec_hidden_dim,
    dec_num_hidden=dec_num_hidden,
    out_dim=num_classes,
).to(device)

prediction_model = PredictionModel(
    num_features=num_features,
    z_dim=z_dim,
    enc_hidden_dim=enc_hidden_dim,
    num_heads=num_heads,
    dec_hidden_dim=dec_hidden_dim,
    dec_num_hidden=dec_num_hidden,
    out_dim=num_classes,
).to(device)

loss_fn = ModelLoss(
    acquisition_model=acquisition_model,
    prediction_model=prediction_model,
    task_type="classification"
).to(device)

In [4]:
# acquisition 모델 학습

num_epochs = 100
lr = 1e-3

optimizer = torch.optim.Adam(
    acquisition_model.parameters(),
    lr=lr
)
scheduler = StepLR(optimizer, step_size=10, gamma=0.1)

best_val_acc = 0.0
best_acquisition_model_state = copy.deepcopy(acquisition_model.state_dict())

for epoch in range(1, num_epochs + 1):
    acquisition_model.train()
    total_loss = 0.0
    total_batches = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        B, D = x.shape

        m_np = sample_mask_uniform_K_per_sample(
            bs=B, d=D,
            min_K=1, max_K=num_features
        )
        m_masked = torch.tensor(m_np, dtype=torch.float32, device=device)
        x_masked = x * m_masked

        out = loss_fn(x_masked, m_masked, y)
        loss = out["acquisition model loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

    avg_loss = total_loss / max(total_batches, 1)

    acquisition_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x_val_batch, y_val_batch in val_loader:
            x_val_batch = x_val_batch.to(device)
            y_val_batch = y_val_batch.to(device)

            m_full_val = torch.ones_like(x_val_batch, device=device)
            logits_val = acquisition_model(x_val_batch, m_full_val)
            preds_val = logits_val.argmax(dim=-1)
            correct += (preds_val == y_val_batch).sum().item()
            total += y_val_batch.size(0)
    val_acc = correct / total if total > 0 else 0.0

    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_acquisition_model_state = copy.deepcopy(acquisition_model.state_dict())

    print(
        f"[Acquisition Model] Epoch {epoch:02d} | loss={avg_loss:.4f} | "
        f"val_acc(acq, full mask)={val_acc:.4f} | best_val_acc={best_val_acc:.4f}"
    )

[Acquisition Model] Epoch 01 | loss=1.4555 | val_acc(acq, full mask)=0.5926 | best_val_acc=0.5926
[Acquisition Model] Epoch 02 | loss=1.1683 | val_acc(acq, full mask)=0.7249 | best_val_acc=0.7249
[Acquisition Model] Epoch 03 | loss=1.0643 | val_acc(acq, full mask)=0.7090 | best_val_acc=0.7249
[Acquisition Model] Epoch 04 | loss=1.0346 | val_acc(acq, full mask)=0.7196 | best_val_acc=0.7249
[Acquisition Model] Epoch 05 | loss=1.0235 | val_acc(acq, full mask)=0.7302 | best_val_acc=0.7302
[Acquisition Model] Epoch 06 | loss=1.0081 | val_acc(acq, full mask)=0.7302 | best_val_acc=0.7302
[Acquisition Model] Epoch 07 | loss=0.9838 | val_acc(acq, full mask)=0.7354 | best_val_acc=0.7354
[Acquisition Model] Epoch 08 | loss=1.0078 | val_acc(acq, full mask)=0.7407 | best_val_acc=0.7407
[Acquisition Model] Epoch 09 | loss=0.9642 | val_acc(acq, full mask)=0.7354 | best_val_acc=0.7407
[Acquisition Model] Epoch 10 | loss=0.9805 | val_acc(acq, full mask)=0.7460 | best_val_acc=0.7460
[Acquisition Model] 

In [5]:
# prediction 모델 학습

num_epochs = 100
lr = 1e-3

optimizer = torch.optim.Adam(
    prediction_model.parameters(),
    lr=lr
)
scheduler = StepLR(optimizer, step_size=10, gamma=0.1)

best_val_acc = 0.0
best_prediction_model_state = copy.deepcopy(prediction_model.state_dict())

for epoch in range(1, num_epochs + 1):
    prediction_model.train()
    total_loss = 0.0
    total_batches = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        B, D = x.shape

        m_np = sample_mask_uniform_K_per_sample(
            bs=B, d=D,
            min_K=1, max_K=num_features
        )
        m_masked = torch.tensor(m_np, dtype=torch.float32, device=device)
        x_masked = x * m_masked

        out = loss_fn(x_masked, m_masked, y)
        loss = out["prediction model loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

    avg_loss = total_loss / max(total_batches, 1)

    prediction_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x_val_batch, y_val_batch in val_loader:
            x_val_batch = x_val_batch.to(device)
            y_val_batch = y_val_batch.to(device)

            m_full_val = torch.ones_like(x_val_batch, device=device)
            logits_val = prediction_model(x_val_batch, m_full_val)
            preds_val = logits_val.argmax(dim=-1)
            correct += (preds_val == y_val_batch).sum().item()
            total += y_val_batch.size(0)
    val_acc = correct / total if total > 0 else 0.0

    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_prediction_model_state = copy.deepcopy(prediction_model.state_dict())

    print(
        f"[Prediction Model] Epoch {epoch:02d} | loss={avg_loss:.4f} | "
        f"val_acc(acq, full mask)={val_acc:.4f} | best_val_acc={best_val_acc:.4f}"
    )

[Prediction Model] Epoch 01 | loss=1.4592 | val_acc(acq, full mask)=0.6720 | best_val_acc=0.6720
[Prediction Model] Epoch 02 | loss=1.1480 | val_acc(acq, full mask)=0.6720 | best_val_acc=0.6720
[Prediction Model] Epoch 03 | loss=1.0370 | val_acc(acq, full mask)=0.6720 | best_val_acc=0.6720
[Prediction Model] Epoch 04 | loss=1.0104 | val_acc(acq, full mask)=0.6931 | best_val_acc=0.6931
[Prediction Model] Epoch 05 | loss=1.0036 | val_acc(acq, full mask)=0.7143 | best_val_acc=0.7143
[Prediction Model] Epoch 06 | loss=0.9493 | val_acc(acq, full mask)=0.7302 | best_val_acc=0.7302
[Prediction Model] Epoch 07 | loss=0.9767 | val_acc(acq, full mask)=0.7090 | best_val_acc=0.7302
[Prediction Model] Epoch 08 | loss=0.9771 | val_acc(acq, full mask)=0.7143 | best_val_acc=0.7302
[Prediction Model] Epoch 09 | loss=0.9486 | val_acc(acq, full mask)=0.7354 | best_val_acc=0.7354
[Prediction Model] Epoch 10 | loss=0.9636 | val_acc(acq, full mask)=0.7513 | best_val_acc=0.7513
[Prediction Model] Epoch 11 | 

In [6]:
# 가중치 저장

acquisition_model.load_state_dict(best_acquisition_model_state)
prediction_model.load_state_dict(best_prediction_model_state)

acquisition_model.eval()
prediction_model.eval()

save_dir = "./checkpoints"
os.makedirs(save_dir, exist_ok=True)

torch.save(best_acquisition_model_state, os.path.join(save_dir, "acquisition_model_metabric.pt"))
torch.save(best_prediction_model_state, os.path.join(save_dir, "prediction_model_metabric.pt"))

print("모델 가중치 저장 완료")

모델 가중치 저장 완료


In [21]:
from sklearn.metrics import accuracy_score

def eval_full_feature(model, X, y, device, batch_size=256):
    model.eval()
    preds_all = []
    y_all = []

    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            # 모든 feature 사용
            m_full = torch.ones_like(xb, device=device)

            logits = model(xb, m_full)
            preds = logits.argmax(dim=-1)

            preds_all.append(preds.cpu())
            y_all.append(yb.cpu())

    preds_all = torch.cat(preds_all).numpy()
    y_all = torch.cat(y_all).numpy()

    acc = accuracy_score(y_all, preds_all)
    return acc


# ====== 실행 ======

acc_acq_full = eval_full_feature(acquisition_model, X_test, y_test, device)
acc_pred_full = eval_full_feature(prediction_model, X_test, y_test, device)

print(f"[Full Feature] Acquisition model ACC: {acc_acq_full:.4f}")
print(f"[Full Feature] Prediction  model ACC: {acc_pred_full:.4f}")


[Full Feature] Acquisition model ACC: 0.7016
[Full Feature] Prediction  model ACC: 0.6963


In [172]:
# Feature Acquisition 성능 확인

result = []

scores = run_feature_acquisition(
    acquisition_model=acquisition_model,
    prediction_model=prediction_model,
    X_test=X_test,
    y_test=y_test,
    alpha=1.0,
    gamma=0.4
)

result.append(scores)

mean_scores = np.mean(scores)
print("Mean acquisition score:", mean_scores)
print(result)

Step 1/12 | ACC: 0.4136
Step 2/12 | ACC: 0.4869
Step 3/12 | ACC: 0.5079
Step 4/12 | ACC: 0.6073
Step 5/12 | ACC: 0.5969
Step 6/12 | ACC: 0.6806
Step 7/12 | ACC: 0.6649
Step 8/12 | ACC: 0.6597
Step 9/12 | ACC: 0.6702
Step 10/12 | ACC: 0.6754
Step 11/12 | ACC: 0.6806
Step 12/12 | ACC: 0.6963
Mean acquisition score: 0.6116928446771379
[[0.41361256544502617, 0.4869109947643979, 0.5078534031413613, 0.6073298429319371, 0.5968586387434555, 0.680628272251309, 0.6649214659685864, 0.6596858638743456, 0.6701570680628273, 0.675392670157068, 0.680628272251309, 0.6963350785340314]]
